In [3]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
import os
import datetime as dt
from datetime import date
import gc

In [5]:
def read_lt_header(state, year):
    """ Read in TAF LT header file """
    ### TAF LT HEADER ###
    # define TAF LT header columns
    header_cols = ['BENE_ID','MSIS_ID','SUBMTG_STATE_CD','CLM_ID',
                          'CLM_TYPE_CD', 'CROSSOVER_CLM_IND', 'SPLIT_CLM_IND',
                          'PYMT_LVL_IND', 'MDCD_ALOWD_AMT', 'MDCD_PD_AMT', 'BENE_LIABILITY_AMT', 
                          'TP_PD_AMT', 'MDCR_PD_AMT','SRVC_BGN_DT','SRVC_END_DT',
                          'MDCD_ACMDTN_PD_AMT','MDCD_ANCLRY_PD_AMT','DAILY_RATE', 
                          'BLG_PRVDR_ID', 'BLG_PRVDR_NPI','BLG_PRVDR_TXNMY_CD','BLG_PRVDR_TYPE_CD']


    
    # read in TAF LT header file (only ID columns) 
    df = dd.read_parquet(f'/{directory_for_lt_parquet_files}/{year}/{state}/taf/lt/lth/parquet', columns=header_cols, engine='pyarrow')


    return df 
    

In [6]:
def format_benemsis(df): 
    """ Format BENE_MSIS ID column in a consistent format""" 
            
        if df.index.name == 'BENE_MSIS': 
            df = df.reset_index()
        
        else: 
            df['BENE_ID'] = df['BENE_ID'].replace('',np.nan)
            df['BENE_MSIS'] = df['BENE_ID'].fillna(df['MSIS_ID'])
            
        df = df.astype({'BENE_MSIS':'str'})
        
        # if BENE_MSIS begin with state, strip the first five characters from BENE_MSIS
        if df['BENE_MSIS'].str.startswith(f'{state}').any().compute():
            df['BENE_MSIS'] = df['BENE_MSIS'].str[5:]
            
        return df

In [7]:
def format_lt_header_cols(df):
    """ Format LT header columns """ 

    numeric_cols = ['MDCD_ALOWD_AMT', 'MDCD_PD_AMT', 'MDCR_PD_AMT', 'MDCD_ACMDTN_PD_AMT', 'MDCD_ANCLRY_PD_AMT','DAILY_RATE']

    for col in numeric_cols: 
        df[col] = dd.to_numeric(df[col], errors='coerce') 

    df = df.astype({'BENE_ID':'str', 
                    'MSIS_ID':'str',
                    'BENE_MSIS':'str',
                    'SUBMTG_STATE_CD':'str',
                    'CLM_ID':'str',
                    'CLM_TYPE_CD':'str',
                    'CROSSOVER_CLM_IND':'str',
                    'SPLIT_CLM_IND':'str',
                    'PYMT_LVL_IND':'str',
                    'MDCD_ALOWD_AMT':'float',
                    'MDCD_PD_AMT':'float',
                    'BENE_LIABILITY_AMT': 'float',
                    'TP_PD_AMT':'float',
                    'MDCR_PD_AMT':'float',
                    'MDCD_ACMDTN_PD_AMT':'float',
                    'MDCD_ANCLRY_PD_AMT':'float',
                    'DAILY_RATE':'float',
                    'BLG_PRVDR_ID':'str',
                    'BLG_PRVDR_NPI':'str', 
                    'BLG_PRVDR_TXNMY_CD':'str',
                    'BLG_PRVDR_TYPE_CD':'str'})
    return df
    


In [8]:
def FFS_claims_only(df):
    """ Limit to FFS claims using the claim type code """ 
    
    df = df.loc[df['CLM_TYPE_CD']=='1']

    return df


In [9]:
def read_taf_de_base(df): 
    """ Read in TAF DE Base file """ 
    
    if (year == 2014) & (state == 'NV'):
        df = dd.read_parquet(f'/gpfs/data/cms-share/data/medicaid/{year}/{state}/taf/de/debse/{state_lower}', columns = ['BENE_ID','MSIS_ID','SUBMTG_STATE_CD','MISG_ELGBLTY_DATA_IND'], engine='pyarrow')

    else: 
        df = dd.read_parquet(f'/gpfs/data/cms-share/data/medicaid/{year}/{state}/taf/de/debse/parquet', columns = ['BENE_ID','MSIS_ID','SUBMTG_STATE_CD','MISG_ELGBLTY_DATA_IND'], engine='pyarrow')
        
    return df 


In [10]:
def read_taf_lt_line(year, state): 
    """ Read in TAF LT line file """ 
    
    state_lower = f'{state}'.lower()  




    taf_lt_line_cols = ['BENE_ID','MSIS_ID','SUBMTG_STATE_CD','SRVC_PRVDR_NPI','CLM_ID', 'REV_CNTR_CD', 'TOS_CD']
                        
                       #  ['LINE_SRVC_BGN_DT','LINE_SRVC_END_DT','LINE_NUM','CLM_NUM_ORIG',
                       #  'LINE_NUM_ORIG','CLM_NUM_ADJ','LINE_NUM_ADJ','LINE_ADJUST_CD', 'LINE_MDCD_ALOWD_AMT', 'LINE_MDCD_PD_AMT',
                       # 'LT_ACCMDTN_HCPCS_RATE'] + service_qty_col


    # read in TAF LT line claims 
    if (year == 2014) & (state == 'AK'): 


        df = dd.read_parquet(f'/gpfs/data/cms-share/data/medicaid/{year}/{state}/taf/lt/ltl/{state_lower}', columns=taf_lt_line_cols, engine='pyarrow')

    else: 
        df = dd.read_parquet(f'/gpfs/data/cms-share/data/medicaid/{year}/{state}/taf/lt/ltl/parquet', columns=taf_lt_line_cols, engine='pyarrow')

    return df


In [11]:
def format_lt_line_cols(df): 
    """ Cast TAF LT line columns to string format" 

    # cast columns to appropriate dtypes
    df = df.astype('str')

    return df


In [12]:
def nf_claims_only(df): 
    """ Limit to nursing facility claims using type of service code or billing provider taxonomy code depending on state """

    # Base condition
    df['nf_claim'] = df['TOS_CD'].isin(['009', '045', '047', '059']).astype(int)

    # Add extra conditions based on state
    if state in ['FL', 'NE', 'NH', 'TX']:
        df['nf_claim'] = df['nf_claim'].where(
            ~((df['nf_claim'] == 0) & 
              (df['TOS_CD'].isnull()) & 
              (df['BLG_PRVDR_TYPE_CD'].isin(['43', '45']))),
            1
        )

    if state in ['CA', 'HI', 'SD']:
        df['nf_claim'] = df['nf_claim'].where(
            ~((df['nf_claim'] == 0) &
              (df['TOS_CD'].isnull()) &
              (df['BLG_PRVDR_TXNMY_CD'].isin(['314000000X', '313M00000X']))),
            1
        )

    if state == 'NM':
        df['nf_claim'] = df['nf_claim'].where(
            ~((df['nf_claim'] == 0) &
              (df['TOS_CD'].isnull()) &
              (df['BLG_PRVDR_TXNMY_CD'].isin(['310000000']))),
            1
        )

        
    return df 

In [13]:
def drop_bedhold_days(df): 
    """ Exclude claims for bedhold days """ 
    df = df.loc[~df['REV_CNTR_CD'].isin(['0180','0182','0183','0184','0185','0189'])]
    
    return df 

In [14]:
def drop_crossover_claims(df): 
    """ Exclude crossover claims using crossover claim indicator """ 

    # ensure that column is in correct format 
    df['CROSSOVER_CLM_IND'] = df['CROSSOVER_CLM_IND'].astype('str')
    
    # value of 0 = NOT a crossover claim 
    df = df.loc[df['CROSSOVER_CLM_IND'] == '0']

    return df 
    

In [15]:
def drop_oneday_claims(df): 
    """ Exclude claims spanning one day or less """ 

    df['SRVC_BGN_DT'] = dd.to_datetime(df['SRVC_BGN_DT'], format='%Y%m%d', errors='coerce')
    df['SRVC_END_DT'] = dd.to_datetime(df['SRVC_END_DT'], format='%Y%m%d', errors='coerce')

    df['DAY_COUNT'] = ((df['SRVC_END_DT'] - df['SRVC_BGN_DT'])/np.timedelta64(1,'D')) + 1

    df = df.loc[df['DAY_COUNT']>1]

    return df 

In [16]:

def format_npi(df): 
    
    """Choose an NPI to identify the facility based on whether the billing provider NPI is available (use service provider NPI if not)."""
    
     df['facility_npi'] = df['BLG_PRVDR_NPI']

    return df 

In [25]:
years = [2016, 2017, 2018] 

test = ['WY']

states = ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 
'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY',
'LA', 'MA', 'MD', 'ME', 'MI', 'MO', 'MS', 'MT', 'NC',
'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 
'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 
'VT', 'WA', 'WI', 'WV', 'WY']

states_2017 = ['AK', 'AL', 'AR', 'AZ', 'CO', 'CT', 'DC', 'DE', 
'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY',
'LA', 'MA', 'MD', 'ME', 'MI', 'MO', 'MS', 'MT', 'NC',
'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 
'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 
'VT', 'WA', 'WI', 'WV', 'WY'] # didn't include california because the directory was named incorrectly and dask can't read the parquet file 

states_2022 = ['AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 
'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY',
'LA', 'MA', 'MD', 'ME', 'MI', 'MO', 'MS', 'MT', 'NC',
'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 
'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 
'VT', 'WA', 'WI', 'WV', 'WY']





In [26]:
### REFACTORED TAF PREP CODE as of 101324 


log = open(f'/{directory_to_store_log_files}/01_prep_taf_files_{date.today()}_delete_me.txt', "w")

for year in [2020,2021]: 
    
    log.write(f'{year}')
    for state in states:

        ### Read in TAF LT header file 
        
        log.write('*********************** \n' + f'{state} \n')
        
        taf_lt_header = read_lt_header(f'{state}', f'{year}')
        
        print(f'{state} read in. # rows: ' + str(taf_lt_header.shape[0].compute()))

        taf_lt_header = format_benemsis(taf_lt_header)
        print('format_bene_msis') 

        taf_lt_header = format_lt_header_cols(taf_lt_header)
        print('format_lt_header_cols')
        
        taf_lt_header = FFS_claims_only(taf_lt_header)
        print('FFS_claims_only') 
        
        # read in TAF LT line columns, which are needed to identify bed hold days and nursing facility claims using TOS_CD, BLG_PRVDR_TXNMY_CD, REV_CNTR_CD
        taf_lt_line = read_taf_lt_line(year, state)
        print('read_taf_lt_line')
        taf_lt_line = format_benemsis(taf_lt_line)
        print('format_bene_msis') 
        taf_lt_line = format_lt_line_cols(taf_lt_line)
        print('format_lt_line_cols') 
                                         
        # check row count, # benes 
        log.write('read in taf line. rows: ' + str(taf_lt_line.shape[0]) + '\n') 
        log.write('number of beneficiaries: ' + (str(len(taf_lt_line['BENE_MSIS'].unique()))) + ('\n'))

        # drop duplicate TAF LT Line rows 
        # print(taf_lt_line.shape[0]) 
        taf_lt_line = taf_lt_line.drop_duplicates(subset=['BENE_MSIS','CLM_ID'], keep='first')
        print('drop lt line duplicates') 

        
        ### MERGE TAF LT HEADER + LINE
        taf_lt_header = taf_lt_header.sort_values(by=['BENE_MSIS', 'CLM_ID'])
        taf_lt_line = taf_lt_line.sort_values(by=['BENE_MSIS', 'CLM_ID'])
        taf_lt_header_line = dd.merge(taf_lt_header, taf_lt_line, how='left', on=['BENE_MSIS','CLM_ID'], indicator='_merge')

        del taf_lt_header 
        del taf_lt_line 
        
        print('merged lt taf header and line') 
        log.write('merge taf lt header and line. _merge value counts: ')
        log.write(str(taf_lt_header_line['_merge'].value_counts()))
        log.write('\n')

        print((taf_lt_header_line['_merge'].value_counts().compute()))

        # drop/rename duplicated columns 
        taf_lt_header_line = taf_lt_header_line.rename(columns={'BENE_ID_x':'BENE_ID', 'MSIS_ID_x':'MSIS_ID','SUBMTG_STATE_CD_x':'SUBMTG_STATE_CD'})
        taf_lt_header_line = taf_lt_header_line.drop(columns=['BENE_ID','MSIS_ID','SPLIT_CLM_IND','BENE_ID_y', 'MSIS_ID_y','SUBMTG_STATE_CD_y', '_merge'])
        log.write('number of claims: ' + str(len(taf_lt_header_line['CLM_ID'].unique())) + '\n')

        # Limit to nursing facility claims only 
        taf_lt_header_line = nf_claims_only(taf_lt_header_line) 
        log.write('limit to nursing facility claims. number of claims: '+ str(len(taf_lt_header_line['CLM_ID'].unique())) + '\n')
        print('nf_claims_only complete') 


        # Exclude bed hold days 
        taf_lt_header_line = drop_bedhold_days(taf_lt_header_line)
        log.write('exclude bed hold days. number of claims: ' + str(len(taf_lt_header_line['CLM_ID'].unique())) + '\n')
        print('drop_bedhold_days complete')

        # Exclude crossover claims 
        taf_lt_header_line = drop_crossover_claims(taf_lt_header_line)
        print('drop_cross_over_claims complete')
        

        
        # Exclude claims with <= 1 day 

        taf_lt_header_line = drop_oneday_claims(taf_lt_header_line)
        # log.write('limit to claims with DAY_COUNT >= 1. Number of claims: ' + str(len(taf_lt_header_line['CLM_ID'].unique())) + '\n')
        print('drop_oneday_claims complete') 


        taf_lt_header_line = format_npi(taf_lt_header_line)                  
        
        # Read out prepared TAF NF claims        
        taf_lt_header_line.to_parquet(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/taf_lt_NF_claims/{year}/{state}', engine='pyarrow', compression='gzip', overwrite=True)
        print(f'{state} dataframe created')
        log.write(f'{state} dataframe created')

        del taf_lt_header_line
        gc.collect()


FileNotFoundError: An error occurred while calling the read_parquet method registered to the pandas backend.
Original Message: /gpfs/data/cms-share/data/medicaid/AR/AK/taf/lt/lth/parquet